<a href="https://colab.research.google.com/github/diddat/Problem-Set-3-Econ_3916_PS3.ipynb/blob/main/Econ_3916_ps3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ECON 3916 Problem Set 3: The Number Nobody Can Reproduce
Tristan Didda

A dashboard reports average revenue per customer in the West as $1,240. Finance reports $812. Both come from the same warehouse, and neither analyst made an arithmetic error. This notebook reproduces both numbers, finds why they differ, and writes the rule that would have caught it.

## Phase 1: Reproduce Both Numbers

### Step 1.1: Build the warehouse
Seed 42 is used throughout Phase 1, so the numbers below are reproducible. Region is assigned independently of order count. Order count is long tailed, and revenue is driven by order count, which is exactly what makes the fan-out matter.

In [ ]:
!pip install duckdb -q

In [ ]:
import duckdb
import numpy as np
import pandas as pd

con = duckdb.connect()

In [ ]:
rng = np.random.default_rng(42)
n_customers = 3000

# Region, assigned independently of order count
u = rng.random(n_customers)
region = np.where(u < 0.70, 'West',
         np.where(u < 0.85, 'East',
         np.where(u < 0.925, 'North', 'South')))

# n_orders = 1 + NegativeBinomial(n=0.8, p=0.12), mean near 7, long right tail
n_orders = 1 + rng.negative_binomial(0.8, 0.12, size=n_customers)

# revenue = 60 * n_orders + Normal(400, 150), floored at 50
revenue = np.maximum(50, 60 * n_orders + rng.normal(400, 150, size=n_customers))

signup_date = (pd.to_datetime('2023-01-01')
               + pd.to_timedelta(rng.integers(0, 730, size=n_customers), unit='D')).date

customers = pd.DataFrame({
    'customer_id': np.arange(1, n_customers + 1),
    'region': region,
    'revenue': revenue,
    'signup_date': signup_date,
})

orders_rows = []
order_id = 1
for customer_id, k in zip(customers['customer_id'], n_orders):
    for _ in range(k):
        orders_rows.append((order_id, customer_id))
        order_id += 1
orders = pd.DataFrame(orders_rows, columns=['order_id', 'customer_id'])

con.register('customers', customers)
con.register('orders', orders)

print('customers rows:', con.execute('SELECT COUNT(*) FROM customers').fetchall()[0][0])
print('West customers:', con.execute("SELECT COUNT(*) FROM customers WHERE region = 'West'").fetchall()[0][0])
print('orders rows:', con.execute('SELECT COUNT(*) FROM orders').fetchall()[0][0])
print('mean orders per customer:', round(float(n_orders.mean()), 2))
print(con.execute('SELECT * FROM customers LIMIT 5').fetchdf())

customers rows: 3000
West customers: 2097
orders rows: 20584
mean orders per customer: 6.86
   customer_id region      revenue signup_date
0            1   East  1091.603668  2024-03-27
1            2   West   791.786388  2024-10-27
2            3  North   803.031871  2024-11-15
3            4   West   503.938050  2024-07-21
4            5   West  1411.171477  2023-12-20


### Step 1.2: Write both queries
The good query averages revenue over the customers table. The bad query averages the same column after joining orders, so every customer is counted once per order they placed.

In [ ]:
good_avg = con.execute("SELECT AVG(revenue) FROM customers WHERE region = 'West'").fetchall()[0][0]
bad_avg = con.execute('''
    SELECT AVG(c.revenue)
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
''').fetchall()[0][0]

good_rows = con.execute("SELECT COUNT(*) FROM customers WHERE region = 'West'").fetchall()[0][0]
bad_rows = con.execute('''
    SELECT COUNT(*)
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
''').fetchall()[0][0]

print(f'Good query (customers only): ${good_avg:.2f}, rows operated on: {good_rows}')
print(f'Bad query (joined to orders): ${bad_avg:.2f}, rows operated on: {bad_rows}')
print(f'Ratio (bad / good): {bad_avg / good_avg:.3f}')
print('Checkpoint: good is in $780 to $845, bad is in $1,160 to $1,320, ratio is in 1.45 to 1.60.')

Good query (customers only): $814.43, rows operated on: 2097
Bad query (joined to orders): $1250.67, rows operated on: 14191
Ratio (bad / good): 1.536
Checkpoint: good is in $780 to $845, bad is in $1,160 to $1,320, ratio is in 1.45 to 1.60.


### Which number is correct, and why
The $812 number is correct, and the $1,240 number is not, for a reason about the unit of observation, not about which query looks tidier.

The question is average revenue per customer, so the unit of observation is the customer, and each customer must count once. The unjoined query does that, it averages over 2,097 West customers. The joined query changes the unit to the customer-order: a customer with 20 orders counts 20 times, and a customer with 1 order counts once. Because revenue rises with order count, the joined average is pulled up by exactly the high order customers who appear most often. It answers a different question, average revenue per order-weighted customer, and that is not the number on the dashboard.

### Step 1.3: The rule
A check that warns when a join changes the row count. It counts rows in the row level version of a query and compares that count to the base table count, before any average is trusted.

In [ ]:
def check_query_rows(con, label, sql, base_count):
    n = con.execute(f'SELECT COUNT(*) FROM ({sql})').fetchall()[0][0]
    if n != base_count:
        print(f'WARNING {label}: row count changed from {base_count} to {n}')
    else:
        print(f'OK {label}: row count unchanged at {n}')

base_west = con.execute("SELECT COUNT(*) FROM customers WHERE region = 'West'").fetchall()[0][0]
check_query_rows(con, 'good query',
                 "SELECT customer_id, revenue FROM customers WHERE region = 'West'",
                 base_west)
check_query_rows(con, 'bad query',
                 "SELECT c.customer_id, c.revenue FROM customers c JOIN orders o ON c.customer_id = o.customer_id WHERE c.region = 'West'",
                 base_west)

OK good query: row count unchanged at 2097
WARNING bad query: row count changed from 2097 to 14191


## Phase 2: The Rows That Disappear

### Step 2.1: Introduce the unknown
Roughly 15 percent of region values are set to NULL, using a separate seed so Phase 1 numbers do not change.

In [ ]:
rng_null = np.random.default_rng(123)
customers_null = customers.copy()
null_mask = rng_null.random(len(customers_null)) < 0.15
customers_null.loc[null_mask, 'region'] = None
con.register('customers_null', customers_null)
print('NULL region rows:', int(null_mask.sum()))
print('Share NULL:', round(float(null_mask.mean()), 3))

NULL region rows: 439
Share NULL: 0.146


### Step 2.2: Lose them, then find them

In [ ]:
lost = con.execute("SELECT COUNT(*) FROM customers_null WHERE region != 'West'").fetchall()[0][0]
intended = con.execute("SELECT COUNT(*) FROM customers_null WHERE region != 'West' OR region IS NULL").fetchall()[0][0]
print(f"WHERE region != 'West': {lost} rows")
print(f"WHERE region != 'West' OR region IS NULL: {intended} rows")
print(f'Difference: {intended - lost} rows')

WHERE region != 'West': 767 rows
WHERE region != 'West' OR region IS NULL: 1206 rows
Difference: 439 rows


Mechanism, in one sentence: in SQL three-valued logic, NULL != 'West' is UNKNOWN, not TRUE, and WHERE keeps only TRUE rows, so the first query silently drops every customer whose region is unknown.

Consequential question: if the missing-region customers are systematically different, newer, smaller, and more likely to churn, then the first query does not just lose rows at random, it removes a particular kind of customer from every regional comparison in the company, so the non-West average is computed only on known-region customers, West versus rest gaps are biased by who is missing rather than by how regions actually perform, and any ranking, budget, or programme decision based on those comparisons will reward or punish regions for their data completeness instead of their revenue.

### Step 2.3: Aggregate without collapsing
One query, with a window function, no join, and no subquery. Each customer keeps their own row, and their region average sits on the same row.

In [ ]:
window_df = con.execute('''
    SELECT customer_id, region, revenue,
           AVG(revenue) OVER (PARTITION BY region) AS region_avg
    FROM customers_null
    ORDER BY customer_id
''').fetchdf()
print('Window query rows:', len(window_df))
print('customers_null rows:', len(customers_null))
print('Row count unchanged:', len(window_df) == len(customers_null))
window_df.head(10)

Window query rows: 3000
customers_null rows: 3000
Row count unchanged: True


,customer_id,region,revenue,region_avg
0,1,East,1091.603668,852.159020
1,2,None,791.786388,794.305458
2,3,North,803.031871,791.297460
3,4,West,503.938050,818.814492
4,5,West,1411.171477,818.814492
5,6,South,381.693166,790.513305
6,7,East,2738.424026,852.159020
7,8,East,554.635105,852.159020
8,9,West,1725.531979,818.814492
9,10,West,1167.000962,818.814492


Note: customers with a NULL region form their own partition under PARTITION BY region, so they are not dropped, and they are compared to the average of the other unknown-region customers, not to West or East.

## Phase 3: The Causal Question

### Step 3.1: The programme evaluation
West got the account-management programme, and West has higher revenue. Here the truth is known by construction: existing account size drives both which region a customer is assigned to and their revenue, and the programme's true effect is small, $50.

Account size is coded 0 for small, 1 for medium, and 2 for large. The chance of being in the West is 0.2 for small accounts, 0.5 for medium accounts, and 0.8 for large accounts. Revenue is $500 plus $400 per size step, plus $50 if the customer got the programme, plus noise.

In [ ]:
rng_c = np.random.default_rng(7)
n_c = 4000
size_idx = rng_c.choice([0, 1, 2], size=n_c, p=[0.5, 0.3, 0.2])
p_west = np.array([0.2, 0.5, 0.8])[size_idx]
west_c = rng_c.random(n_c) < p_west
programme_c = west_c.astype(int)
causal_revenue = 500 + 400 * size_idx + 50 * programme_c + rng_c.normal(0, 100, size=n_c)

causal = pd.DataFrame({
    'size_idx': size_idx,
    'west': west_c,
    'programme': programme_c,
    'revenue': causal_revenue,
})
con.register('causal', causal)

naive = con.execute('''
    SELECT AVG(CASE WHEN west THEN revenue END) - AVG(CASE WHEN NOT west THEN revenue END)
    FROM causal
''').fetchall()[0][0]

adjusted = con.execute('''
    SELECT SUM(w * diff) / SUM(w) FROM (
        SELECT size_idx, COUNT(*) AS w,
               AVG(CASE WHEN west THEN revenue END) - AVG(CASE WHEN NOT west THEN revenue END) AS diff
        FROM causal
        GROUP BY size_idx
    )
''').fetchall()[0][0]

print(f'True programme effect: $50')
print(f'Naive West minus East difference: ${naive:.1f}')
print(f'Adjusted difference, within account-size groups: ${adjusted:.1f}')
print(con.execute('''
    SELECT size_idx, COUNT(*) AS n,
           ROUND(AVG(CASE WHEN west THEN revenue END), 1) AS west_avg,
           ROUND(AVG(CASE WHEN NOT west THEN revenue END), 1) AS east_avg
    FROM causal GROUP BY size_idx ORDER BY size_idx
''').fetchdf())

True programme effect: $50
Naive West minus East difference: $363.3
Adjusted difference, within account-size groups: $48.3
   size_idx     n  west_avg  east_avg
0         0  2013     544.6     502.6
1         1  1167     953.8     898.3
2         2   820    1350.8    1297.3


Report against the truth: the naive comparison, $363.3, overstates the true $50 effect by about $313, because West has far more large accounts and large accounts earn more with or without the programme. The adjusted comparison, $48.3, is close to the true $50, because it compares West and East customers within the same account-size group and then averages those within-group differences. The West revenue advantage is mostly account size, not the programme.

### Step 3.2: The control that ruins it
The director suggests controlling for number of support tickets, because tickets look like engagement. Here tickets are built as a result of both the programme and revenue: tickets rise with the programme, and tickets rise with revenue. The score below is split at its median into high and low tickets, and the comparison is then done within account-size and ticket groups.

In [ ]:
rng_t = np.random.default_rng(11)
tickets_score = 2.0 * programme_c + 0.004 * (causal_revenue - 800) + rng_t.normal(0, 1.0, size=n_c)
high_tickets = (tickets_score > np.median(tickets_score)).astype(int)

coll = pd.DataFrame({
    'size_idx': size_idx,
    'west': west_c,
    'programme': programme_c,
    'revenue': causal_revenue,
    'high_tickets': high_tickets,
})
con.register('coll', coll)

adj_size_only = con.execute('''
    SELECT SUM(w * diff) / SUM(w) FROM (
        SELECT size_idx, COUNT(*) AS w,
               AVG(CASE WHEN west THEN revenue END) - AVG(CASE WHEN NOT west THEN revenue END) AS diff
        FROM coll GROUP BY size_idx
    )
''').fetchall()[0][0]

adj_size_tickets = con.execute('''
    SELECT SUM(w * diff) / SUM(w) FROM (
        SELECT size_idx, high_tickets, COUNT(*) AS w,
               AVG(CASE WHEN west THEN revenue END) - AVG(CASE WHEN NOT west THEN revenue END) AS diff
        FROM coll GROUP BY size_idx, high_tickets
    )
''').fetchall()[0][0]

print(f'True programme effect: $50')
print(f'Adjusted for account size only: ${adj_size_only:.1f}')
print(f'Adjusted for account size and tickets: ${adj_size_tickets:.1f}')

True programme effect: $50
Adjusted for account size only: $48.3
Adjusted for account size and tickets: $10.3


### Paragraph for the director
Dear Director, I would not add support tickets as a control, even though it sounds sensible. Tickets are not a background feature of a customer, they are something that happens after the programme and after revenue starts to come in: the programme itself creates more contact and more tickets, and larger, higher revenue accounts also generate more tickets because they use the product more and have more to ask about. If we only compare customers with the same number of tickets, we are no longer comparing like with like. A West customer who reached a high ticket count mainly because of the programme will be put next to an East customer who reached the same ticket count mainly because they are already a large, high revenue account, and that comparison makes the programme look smaller than it is, or even useless, when the real problem is the comparison we forced. The right comparison is customers who looked similar before the programme started, by account size, not customers who ended up looking similar afterwards. On that basis, the programme looks like a small help, not the main reason West revenue is higher.

## Phase 4: AI Expansion, The Co-Pilot Era
Status: AI usage AUTHORISED and REQUIRED for this phase. The P.R.I.M.E. framework was used: Prep, Request, Iterate, Mechanism Check, Evaluate.

# AI Transcript
Model: Claude
Date: 1 November 2026

## Prompt sent
[Prep] Act as an analytics engineer who reviews SQL before it reaches a dashboard.

[Request] Write a Python function review_query(con, sql) that runs against DuckDB and returns a list of warnings for a given SELECT. It should flag at minimum: a join that changes the row count, a filter on a nullable column that will silently drop unknowns, and an aggregate computed over a fanned-out result.

[Iterate] duckdb and pandas only. Reuse my table names, customers and orders. Return structured warnings, not printed text, so I can test it.

[Mechanism Check] Add comments explaining how you detect fan-out without running the full join twice, and why three-valued logic makes the null filter a correctness issue rather than a style issue.

[Evaluate] Run it against both of my Phase 1 queries and tell me whether it would have caught the $1,240 before it reached the board. Run it on your Phase 1 and Phase 2 queries. Report what it caught and what it missed. A tool that misses something is a useful finding if you say so.

Return only the Python code for review_query, with its comments, plus a very short note at the end (no more than 3 sentences) saying what it would catch. Do not run it against any real student tables, and do not invent test results. Use table names customers (customer_id, region, revenue, signup_date) and orders (order_id, customer_id).

## Model reply

```python
import re

import pandas as pd


def review_query(con, sql):
    """Review one SELECT against DuckDB and return structured warnings.

    Parameters
    ----------
    con : duckdb.DuckDBPyConnection
        An open DuckDB connection holding ``customers`` and ``orders``.
    sql : str
        A single SELECT statement to review.

    Returns
    -------
    list[dict]
        Structured warnings, not printed text, so a caller can test them.
        Each warning has ``code``, ``severity``, ``message`` and ``detail``.
    """
    warnings = []

    if not isinstance(sql, str) or not sql.strip():
        return [{
            "code": "EMPTY_QUERY",
            "severity": "high",
            "message": "No SQL was supplied to review.",
            "detail": {},
        }]

    text = sql.strip()
    lower = text.lower()

    # Only SELECTs are reviewed; anything else cannot be safely probed
    # as a dashboard query, and running it could have side effects.
    if not re.match(r"(?s)^\s*(with\b.*?\bselect\b|select\b)", lower):
        return [{
            "code": "NOT_SELECT",
            "severity": "high",
            "message": "Only SELECT statements are reviewed.",
            "detail": {},
        }]

    # ---- Schema: which columns are nullable? ---------------------------
    # information_schema is used (via pandas) so the reviewer knows the
    # actual tables, rather than assuming a column can or cannot be NULL.
    schema = {}
    try:
        schema_df = con.execute(
            "SELECT table_name, column_name, is_nullable "
            "FROM information_schema.columns "
            "WHERE table_name IN ('customers', 'orders')"
        ).fetchdf()
        for row in schema_df.itertuples(index=False):
            schema.setdefault(str(row.table_name).lower(), {})[
                str(row.column_name).lower()
            ] = str(row.is_nullable).upper() == "YES"
    except Exception as exc:  # Keep reviewing what can be checked.
        warnings.append({
            "code": "SCHEMA_CHECK_FAILED",
            "severity": "low",
            "message": "Could not read column nullability from the schema.",
            "detail": {"error": str(exc)},
        })

    tables_in_query = [t for t in ("customers", "orders")
                       if re.search(rf"\b{t}\b", lower)]

    # ---- Table aliases, e.g. FROM customers c JOIN orders o -----------
    alias_to_table = {"customers": "customers", "orders": "orders"}
    keywords = {
        "where", "group", "order", "limit", "join", "on", "left",
        "right", "inner", "full", "cross", "using", "having",
    }
    for match in re.finditer(
        r"\b(?:from|join)\s+(customers|orders)"
        r"(?:\s+as\s+([a-z_][a-z0-9_]*)|\s+([a-z_][a-z0-9_]*))?",
        lower,
    ):
        alias = match.group(2) or match.group(3)
        if alias and alias not in keywords:
            alias_to_table[alias] = match.group(1)

    # ---- Fan-out detection, without running the full join twice -------
    # Mechanism check: do NOT materialise the user's join, and do NOT run
    # it once for a row count and again for the result. Instead, probe
    # each side of the join on its own with one cheap aggregate:
    #   COUNT(*) vs COUNT(DISTINCT join_key) on that table alone.
    # If the right-hand (or left-hand) table has more rows than distinct
    # join keys, some keys repeat, so joining on that key must duplicate
    # rows from the other side for those keys. That is exactly the
    # fan-out that makes AVG(customers.revenue) after JOIN orders
    # weight a customer once per order instead of once per customer.
    # This probe is a warning about structure, not a proof of the size
    # of the bias for a filtered query, because a WHERE clause could
    # in principle remove the duplicated keys.
    join_pairs = []
    for match in re.finditer(
        r"([a-z_][a-z0-9_]*)\.([a-z_][a-z0-9_]*)\s*=\s*"
        r"([a-z_][a-z0-9_]*)\.([a-z_][a-z0-9_]*)",
        lower,
    ):
        left_alias, left_col, right_alias, right_col = match.groups()
        left_table = alias_to_table.get(left_alias)
        right_table = alias_to_table.get(right_alias)
        if left_table and right_table:
            join_pairs.append((left_table, left_col, right_table, right_col))

    fanout_details = []
    for left_table, left_col, right_table, right_col in join_pairs:
        for table, col in ((left_table, left_col),
                           (right_table, right_col)):
            try:
                total, distinct_keys = con.execute(
                    f'SELECT COUNT(*), COUNT(DISTINCT "{col}") '
                    f'FROM "{table}"'
                ).fetchone()
            except Exception:
                continue
            if total is not None and distinct_keys is not None \
                    and total > distinct_keys:
                fanout_details.append({
                    "table": table,
                    "join_column": col,
                    "rows": int(total),
                    "distinct_join_keys": int(distinct_keys),
                })

    has_join = "join" in lower
    if has_join and fanout_details:
        warnings.append({
            "code": "JOIN_FANOUT_ROW_COUNT_CHANGE",
            "severity": "high",
            "message": (
                "Join can change the row count: a join key repeats on "
                "one side, so rows from the other side will be duplicated."
            ),
            "detail": {"duplicated_keys": fanout_details},
        })
    elif re.search(r"\bcross\s+join\b", lower):
        warnings.append({
            "code": "JOIN_FANOUT_ROW_COUNT_CHANGE",
            "severity": "high",
            "message": "CROSS JOIN multiplies rows from both sides.",
            "detail": {},
        })

    # ---- Nullable filter that silently drops unknowns ------------------
    # Mechanism check: this is a correctness issue, not a style issue,
    # because of SQL three-valued logic. A comparison involving NULL
    # is neither TRUE nor FALSE, it is UNKNOWN, and WHERE keeps only
    # TRUE rows. So WHERE region != 'West' does not return "everyone
    # who is not West", it returns "everyone known not to be West"
    # and silently discards every row whose region is unknown, with no
    # error. Only negation filters are flagged here: a positive
    # filter such as region = 'West' already excludes unknowns by
    # definition, whereas a complement filter is the one whose intended
    # meaning is changed by UNKNOWN.
    where_match = re.search(
        r"\bwhere\b(.*?)(?:\bgroup\s+by\b|\border\s+by\b|\blimit\b|"
        r"\bhaving\b|\bwindow\b|\bqualify\b|$)",
        lower,
        flags=re.S,
    )
    if where_match:
        where_text = where_match.group(1)
        for match in re.finditer(
            r"(?:([a-z_][a-z0-9_]*)\.)?([a-z_][a-z0-9_]*)\s*"
            r"(<>|!=|\bnot\s+in\b|\bnot\s+like\b)",
            where_text,
        ):
            qualifier, col, _op = match.groups()
            if qualifier:
                candidate_tables = [alias_to_table[qualifier]] \
                    if qualifier in alias_to_table else []
            else:
                candidate_tables = [
                    t for t in tables_in_query if col in schema.get(t, {})
                ]
            for table in candidate_tables:
                if schema.get(table, {}).get(col):
                    warnings.append({
                        "code": "NULL_FILTER_DROPS_UNKNOWNS",
                        "severity": "high",
                        "message": (
                            f"Negation filter on nullable column "
                            f"{table}.{col} silently drops rows where "
                            f"{col} is NULL, because NULL comparisons "
                            f"are UNKNOWN and WHERE keeps only TRUE."
                        ),
                        "detail": {"table": table, "column": col},
                    })

    # ---- Aggregate computed over a fanned-out result -------------------
    select_part_match = re.search(
        r"\bselect\b(.*?)\bfrom\b", lower, flags=re.S
    )
    select_part = select_part_match.group(1) if select_part_match else lower
    aggregates = re.findall(
        r"\b(avg|sum|count|min|max)\s*\(\s*(distinct\s+)?([^)]*)\)",
        select_part,
    )
    non_distinct_aggs = [fn for fn, distinct, _arg in aggregates
                         if not distinct]
    if non_distinct_aggs and fanout_details:
        warnings.append({
            "code": "AGGREGATE_OVER_FANOUT",
            "severity": "high",
            "message": (
                "Aggregate is computed over a joined result whose rows "
                "have been duplicated, so each duplicated row is weighted "
                "once per join match instead of once per original row."
            ),
            "detail": {
                "aggregates": non_distinct_aggs,
                "duplicated_keys": fanout_details,
            },
        })

    return warnings
```

It would catch the Phase 1 fan-out, because joining customers to orders duplicates customers with many orders and an average over that joined result weights them repeatedly, and it would catch a WHERE region != 'West' filter on a nullable region column. It would not by itself prove the exact board figure, because it flags the structure of the query rather than recomputing the arithmetic.

### Changes I made to the model reply
One line on each change, as the assignment asks:
- Generalised table parsing to any table name in FROM or JOIN, so the Phase 2 table customers_null is checked, where the original only looked for customers and orders.
- Suppressed the null filter warning when the WHERE clause already includes IS NULL for that column, so the intended Phase 2 query is not flagged.
- Added a check for actual NULL values in the data, in addition to schema nullability, so a column that contains NULLs is flagged even if the schema is unclear.
- Kept the structured warning format, and kept the fan-out probe as a per-table COUNT versus COUNT DISTINCT check, so the join is not materialised twice.

### Task 4.1: The query reviewer, corrected version

In [ ]:
import re

def review_query(con, sql):
    # Returns structured warnings, not printed text, so a caller can test them.
    warnings = []
    if not isinstance(sql, str) or not sql.strip():
        return [{'code': 'EMPTY_QUERY', 'severity': 'high',
                 'message': 'No SQL was supplied.', 'detail': {}}]
    text = sql.strip()
    lower = text.lower()
    if not re.match(r'(?s)^\s*(with\b.*?\bselect\b|select\b)', lower):
        return [{'code': 'NOT_SELECT', 'severity': 'high',
                 'message': 'Only SELECT statements are reviewed.', 'detail': {}}]

    # Tables in FROM or JOIN, any name, not only customers and orders.
    tables = []
    for m in re.finditer(r'\b(?:from|join)\s+([a-z_][a-z0-9_]*)', lower):
        if m.group(1) not in tables:
            tables.append(m.group(1))
    alias_to_table = {}
    for m in re.finditer(r'\b(?:from|join)\s+([a-z_][a-z0-9_]*)(?:\s+as\s+([a-z_][a-z0-9_]*)|\s+([a-z_][a-z0-9_]*))?', lower):
        alias = m.group(2) or m.group(3)
        if alias and alias not in {'where', 'group', 'order', 'limit', 'join', 'on', 'left', 'right', 'inner', 'full', 'cross', 'using', 'having'}:
            alias_to_table[alias] = m.group(1)
    for t in tables:
        alias_to_table.setdefault(t, t)

    # Schema nullability for those tables.
    nullable = {}
    for t in tables:
        try:
            df = con.execute(f"SELECT column_name, is_nullable FROM information_schema.columns WHERE table_name = '{t}'").fetchdf()
            for row in df.itertuples(index=False):
                nullable[(t, str(row.column_name).lower())] = str(row.is_nullable).upper() == 'YES'
        except Exception:
            pass

    # Fan-out detection, without running the full join twice:
    # probe each join side on its own with COUNT(*) versus COUNT(DISTINCT join_key).
    # If a side has more rows than distinct keys, joining on that key must duplicate
    # rows from the other side. This is a structural warning, not a proof of bias
    # size for a filtered query, because a WHERE clause could remove duplicated keys.
    join_pairs = []
    for m in re.finditer(r'([a-z_][a-z0-9_]*)\.([a-z_][a-z0-9_]*)\s*=\s*([a-z_][a-z0-9_]*)\.([a-z_][a-z0-9_]*)', lower):
        la, lc, ra, rc = m.groups()
        if la in alias_to_table and ra in alias_to_table:
            join_pairs.append((alias_to_table[la], lc, alias_to_table[ra], rc))
    fanout_details = []
    for lt, lc, rt, rc in join_pairs:
        for table, col in ((lt, lc), (rt, rc)):
            try:
                total, distinct_keys = con.execute(f'SELECT COUNT(*), COUNT(DISTINCT "{col}") FROM "{table}"').fetchall()[0]
            except Exception:
                continue
            if total is not None and distinct_keys is not None and total > distinct_keys:
                fanout_details.append({'table': table, 'join_column': col, 'rows': int(total), 'distinct_join_keys': int(distinct_keys)})
    has_join = 'join' in lower
    if has_join and fanout_details:
        warnings.append({'code': 'JOIN_FANOUT_ROW_COUNT_CHANGE', 'severity': 'high',
            'message': 'Join can change the row count: a join key repeats on one side, so rows from the other side will be duplicated.',
            'detail': {'duplicated_keys': fanout_details}})

    # Nullable filter: three-valued logic makes this a correctness issue, not style.
    # A comparison with NULL is UNKNOWN, not TRUE, and WHERE keeps only TRUE rows,
    # so a negation filter silently drops unknown rows instead of returning the complement set.
    where_match = re.search(r'\bwhere\b(.*?)(?:\bgroup\s+by\b|\border\s+by\b|\blimit\b|\bhaving\b|\bwindow\b|\bqualify\b|$)', lower, flags=re.S)
    if where_match:
        where_text = where_match.group(1)
        for m in re.finditer(r'(?:([a-z_][a-z0-9_]*)\.)?([a-z_][a-z0-9_]*)\s*(<>|!=|\bnot\s+in\b|\bnot\s+like\b)', where_text):
            qualifier, col, _op = m.groups()
            if qualifier:
                cand = [alias_to_table[qualifier]] if qualifier in alias_to_table else []
            else:
                cand = [t for t in tables if (t, col) in nullable]
            for table in cand:
                if re.search(rf'(?:[a-z_][a-z0-9_]*\.)?{col}\s+is\s+null', where_text):
                    continue
                is_null = nullable.get((table, col), False)
                try:
                    has_null = con.execute(f'SELECT COUNT(*) FROM "{table}" WHERE "{col}" IS NULL').fetchall()[0][0] > 0
                except Exception:
                    has_null = False
                if is_null or has_null:
                    warnings.append({'code': 'NULL_FILTER_DROPS_UNKNOWNS', 'severity': 'high',
                        'message': f'Negation filter on nullable column {table}.{col} silently drops rows where {col} is NULL, because NULL comparisons are UNKNOWN and WHERE keeps only TRUE.',
                        'detail': {'table': table, 'column': col}})

    select_part = re.search(r'\bselect\b(.*?)\bfrom\b', lower, flags=re.S)
    select_part = select_part.group(1) if select_part else lower
    aggs = re.findall(r'\b(avg|sum|count|min|max)\s*\(\s*(distinct\s+)?([^)]*)\)', select_part)
    non_distinct = [fn for fn, distinct, _arg in aggs if not distinct]
    if non_distinct and fanout_details:
        warnings.append({'code': 'AGGREGATE_OVER_FANOUT', 'severity': 'high',
            'message': 'Aggregate is computed over a joined result whose rows have been duplicated, so each duplicated row is weighted once per join match instead of once per original row.',
            'detail': {'aggregates': non_distinct, 'duplicated_keys': fanout_details}})
    return warnings

In [ ]:
queries = {
    'Phase 1 good': "SELECT AVG(revenue) FROM customers WHERE region = 'West'",
    'Phase 1 bad': "SELECT AVG(c.revenue) FROM customers c JOIN orders o ON c.customer_id = o.customer_id WHERE c.region = 'West'",
    'Phase 2 bad': "SELECT COUNT(*) FROM customers_null WHERE region != 'West'",
    'Phase 2 good': "SELECT COUNT(*) FROM customers_null WHERE region != 'West' OR region IS NULL",
    'Phase 2 window': "SELECT customer_id, region, revenue, AVG(revenue) OVER (PARTITION BY region) AS region_avg FROM customers_null",
}
for label, q in queries.items():
    print(label)
    print(review_query(con, q))
    print()

Phase 1 good
[]

Phase 1 bad
[{'code': 'JOIN_FANOUT_ROW_COUNT_CHANGE', 'severity': 'high', 'message': 'Join can change the row count: a join key repeats on one side, so rows from the other side will be duplicated.', 'detail': {'duplicated_keys': [{'table': 'orders', 'join_column': 'customer_id', 'rows': 20584, 'distinct_join_keys': 3000}]}}, {'code': 'AGGREGATE_OVER_FANOUT', 'severity': 'high', 'message': 'Aggregate is computed over a joined result whose rows have been duplicated, so each duplicated row is weighted once per join match instead of once per original row.', 'detail': {'aggregates': ['avg'], 'duplicated_keys': [{'table': 'orders', 'join_column': 'customer_id', 'rows': 20584, 'distinct_join_keys': 3000}]}}]

Phase 2 bad
[{'code': 'NULL_FILTER_DROPS_UNKNOWNS', 'severity': 'high', 'message': 'Negation filter on nullable column customers_null.region silently drops rows where region is NULL, because NULL comparisons are UNKNOWN and WHERE keeps only TRUE.', 'detail': {'table': 'c

### What the reviewer caught, and what it missed
Caught: the Phase 1 bad query gets both a join fan-out warning and an aggregate over fan-out warning, so yes, it would have caught the $1,240 before it reached the board. The Phase 2 bad query gets a null filter warning. The Phase 1 good query, the Phase 2 intended query, and the window query get no warnings, which is the correct silent behaviour for those three.

Missed: the original model version missed the Phase 2 bad query entirely, because it only looked for tables named customers and orders, and customers_null was not in that list, which is why the generalisation above was needed. Even the corrected version still misses fan-out hidden in subqueries or common table expressions, joins on expressions rather than simple column equals column, and negation written as NOT (region = 'West'). It also still flags a join structurally even when the aggregate is COUNT DISTINCT, because the join flag does not check whether the aggregate itself is distinct safe, even though the aggregate flag correctly does. It cannot judge whether a control is a confounder or a post-treatment measure, and it cannot tell whether the unit of observation is right, it can only warn that rows were duplicated.

### Task 4.2: One-page writeup, 182 words
Dear Director,
The $812 figure is the right one. It averages revenue once per West customer. The $1,240 figure averages the same revenue after joining orders, so a customer with 20 orders counts 20 times, and high order customers pull the average up. The gap is a counting problem, not a pricing or finance error.
On the programme, the West advantage is mostly not the programme. In a comparison that accounts for existing account size, the West advantage falls from about $363 to about $48, close to a small true effect. The programme may help a little, but it does not explain most of the gap, because West started with larger accounts.
Do not add support tickets as a control. Tickets rise after the programme and with higher revenue, so holding tickets fixed compares unlike customers and shrinks the estimate.
Before any number goes on a slide, check the row count before and after every join, state the unit being averaged, handle unknown values explicitly instead of dropping them, and run the query reviewer. Those four checks would have caught this one.